In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip -q install librosa soundfile audiomentations tqdm scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 28.3 MB/s eta 0:00:00


In [3]:
import os
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import librosa

from pathlib import Path
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

BASE_PROJECT = Path("/content/drive/MyDrive/New Jurnal Cross")

CSV_DIR = BASE_PROJECT / "processed_intra_csv"
OUT_DIR = BASE_PROJECT / "processed_intra_features_hc_noaug"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "sample_rate": 16000,
    "duration": 4.0,

    "n_mfcc": 40,
    "n_mels": 128,
    "n_fft": 1024,
    "hop_length": 256,

    "aug_n_copies": 0,

    "unified_emotions": [
        "angry",
        "disgust",
        "fear",
        "happy",
        "neutral",
        "sad",
    ],

    "csv_files": {
        "emodb": CSV_DIR / "split_emodb_6class_optimized.csv",
        "ravdess": CSV_DIR / "split_ravdess_6class_optimized.csv",
        "resd": CSV_DIR / "split_resd_6class_optimized.csv",
    }
}

LABEL_TO_ID = {label: i for i, label in enumerate(CONFIG["unified_emotions"])}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

print("CSV_DIR:", CSV_DIR)
print("OUT_DIR:", OUT_DIR)

for ds, path in CONFIG["csv_files"].items():
    print(f"{ds:8s}", path.exists(), path)

CSV_DIR: /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv
OUT_DIR: /content/drive/MyDrive/New Jurnal Cross/processed_intra_features_hc_noaug
emodb    True /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/split_emodb_6class_optimized.csv
ravdess  True /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/split_ravdess_6class_optimized.csv
resd     True /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/split_resd_6class_optimized.csv


In [4]:
try:
    from audiomentations import (
        Compose,
        AddGaussianNoise,
        TimeStretch,
        PitchShift,
        Shift
    )
    HAS_AUDIOMENTATIONS = True
    print("✅ audiomentations tersedia")
except ImportError:
    HAS_AUDIOMENTATIONS = False
    print("⚠️ audiomentations tidak tersedia")


def build_augmenter():
    if not HAS_AUDIOMENTATIONS:
        return None

    return Compose([
        AddGaussianNoise(
            min_amplitude=0.001,
            max_amplitude=0.015,
            p=0.50
        ),
        TimeStretch(
            min_rate=0.85,
            max_rate=1.15,
            p=0.50
        ),
        PitchShift(
            min_semitones=-3,
            max_semitones=3,
            p=0.50
        ),
        Shift(
            min_shift=-0.20,
            max_shift=0.20,
            p=0.30
        ),
    ])


augmenter = build_augmenter()
print("Augmenter:", augmenter)

✅ audiomentations tersedia
Augmenter: Compose([
  AddGaussianNoise(p=0.5, min_amplitude=0.001, max_amplitude=0.015),
  TimeStretch(p=0.5, min_rate=0.85, max_rate=1.15, leave_length_unchanged=True, method='signalsmith_stretch'),
  PitchShift(p=0.5, min_semitones=-3, max_semitones=3, method='signalsmith_stretch'),
  Shift(p=0.3, min_shift=-0.2, max_shift=0.2, shift_unit='fraction', rollover=True, fade_duration=0.005),
], p=1.0)


In [5]:
def load_audio(path, sr=16000, duration=4.0):
    """
    Load audio mono, resample ke sr, lalu crop/pad ke durasi tetap.
    Crop menggunakan segmen tengah untuk mengurangi silence awal/akhir.
    """
    audio, _ = librosa.load(path, sr=sr, mono=True)

    target_len = int(sr * duration)

    if len(audio) >= target_len:
        start = (len(audio) - target_len) // 2
        audio = audio[start:start + target_len]
    else:
        pad_len = target_len - len(audio)

        if len(audio) > 1:
            audio = np.pad(audio, (0, pad_len), mode="reflect")
        else:
            audio = np.pad(audio, (0, pad_len), mode="constant")

    return audio.astype(np.float32)

In [6]:
def cmvn_per_utterance(mfcc):
    """
    Cepstral Mean Variance Normalization per utterance.
    Shape input: [n_mfcc, time]
    """
    mean = mfcc.mean(axis=1, keepdims=True)
    std = mfcc.std(axis=1, keepdims=True) + 1e-8
    return (mfcc - mean) / std


def safe_stats(x):
    """
    Return mean dan std dari feature sequence.
    """
    x = np.asarray(x)

    if x.ndim == 1:
        x = x.reshape(1, -1)

    mean = np.nanmean(x, axis=1)
    std = np.nanstd(x, axis=1)

    mean = np.nan_to_num(mean, nan=0.0, posinf=0.0, neginf=0.0)
    std = np.nan_to_num(std, nan=0.0, posinf=0.0, neginf=0.0)

    return mean, std


def extract_prosodic(audio, sr, hop_length):
    """
    Prosodic features:
    - F0 mean, std, min, max, range
    - F0 jitter approximation
    - RMS mean, std, max
    - ZCR mean

    Total: 10 dim
    """
    feats = []

    try:
        f0, voiced_flag, _ = librosa.pyin(
            audio,
            fmin=50,
            fmax=500,
            sr=sr,
            hop_length=hop_length
        )

        if voiced_flag is not None and np.any(voiced_flag):
            f0_voiced = f0[voiced_flag]
            f0_voiced = f0_voiced[~np.isnan(f0_voiced)]
        else:
            f0_voiced = np.array([], dtype=np.float32)

        if len(f0_voiced) > 0:
            f0_mean = np.mean(f0_voiced)
            f0_std = np.std(f0_voiced)
            f0_min = np.min(f0_voiced)
            f0_max = np.max(f0_voiced)
            f0_range = f0_max - f0_min

            if len(f0_voiced) > 1:
                jitter = np.mean(np.abs(np.diff(f0_voiced)))
            else:
                jitter = 0.0

            feats += [
                f0_mean,
                f0_std,
                f0_min,
                f0_max,
                f0_range,
                jitter,
            ]
        else:
            feats += [0.0] * 6

    except Exception:
        feats += [0.0] * 6

    rms = librosa.feature.rms(y=audio, hop_length=hop_length)[0]
    feats += [
        float(np.mean(rms)),
        float(np.std(rms)),
        float(np.max(rms)),
    ]

    zcr = librosa.feature.zero_crossing_rate(y=audio, hop_length=hop_length)[0]
    feats.append(float(np.mean(zcr)))

    feats = np.array(feats, dtype=np.float32)
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

    return feats

In [7]:
def extract_handcrafted(audio, cfg):
    """
    Handcrafted acoustic-prosodic feature vector tanpa duplikasi.

    Komponen:
    - MFCC CMVN mean+std          : 40*2 = 80
    - Delta MFCC mean+std         : 40*2 = 80
    - Delta-delta MFCC mean+std   : 40*2 = 80
    - Chroma mean+std             : 12*2 = 24
    - Log Mel mean+std            : 128*2 = 256
    - Spectral Contrast mean+std  : 7*2 = 14
    - Spectral Rolloff mean+std   : 1*2 = 2
    - Spectral Bandwidth mean+std : 1*2 = 2
    - Prosodic                    : 10

    Prosodic berisi:
    - F0 mean, std, min, max, range
    - F0 jitter approximation
    - RMS mean, std, max
    - ZCR mean

    Total: 548 dim
    """
    sr = cfg["sample_rate"]
    hop = cfg["hop_length"]
    n_fft = cfg["n_fft"]

    features = []

    # MFCC
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=cfg["n_mfcc"],
        hop_length=hop,
        n_fft=n_fft
    )
    mfcc = cmvn_per_utterance(mfcc)

    m, s = safe_stats(mfcc)
    features += [m, s]

    # Delta MFCC
    delta = librosa.feature.delta(mfcc)
    m, s = safe_stats(delta)
    features += [m, s]

    # Delta-delta MFCC
    delta2 = librosa.feature.delta(mfcc, order=2)
    m, s = safe_stats(delta2)
    features += [m, s]

    # Chroma
    chroma = librosa.feature.chroma_stft(
        y=audio,
        sr=sr,
        hop_length=hop,
        n_fft=n_fft
    )
    m, s = safe_stats(chroma)
    features += [m, s]

    # Log Mel Spectrogram
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=cfg["n_mels"],
        hop_length=hop,
        n_fft=n_fft
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    m, s = safe_stats(mel_db)
    features += [m, s]

    # Spectral Contrast
    spectral_contrast = librosa.feature.spectral_contrast(
        y=audio,
        sr=sr,
        hop_length=hop,
        n_fft=n_fft
    )
    m, s = safe_stats(spectral_contrast)
    features += [m, s]

    # Spectral Rolloff
    rolloff = librosa.feature.spectral_rolloff(
        y=audio,
        sr=sr,
        hop_length=hop,
        n_fft=n_fft
    )
    m, s = safe_stats(rolloff)
    features += [m, s]

    # Spectral Bandwidth
    bandwidth = librosa.feature.spectral_bandwidth(
        y=audio,
        sr=sr,
        hop_length=hop,
        n_fft=n_fft
    )
    m, s = safe_stats(bandwidth)
    features += [m, s]

    # Prosodic: F0 + RMS + ZCR
    pros = extract_prosodic(audio, sr, hop)
    features.append(pros)

    x = np.concatenate(features).astype(np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    return x

In [8]:
def extract_original_features(df, cfg, desc="Extract original"):
    X = []
    valid_rows = []
    errors = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        try:
            audio = load_audio(
                row["filepath"],
                sr=cfg["sample_rate"],
                duration=cfg["duration"]
            )

            feat = extract_handcrafted(audio, cfg)

            X.append(feat)
            valid_rows.append(row.to_dict())

        except Exception as e:
            errors.append({
                "idx": idx,
                "filepath": row["filepath"],
                "error": str(e)
            })

    X = np.asarray(X, dtype=np.float32)
    meta_valid = pd.DataFrame(valid_rows)

    return X, meta_valid, pd.DataFrame(errors)

In [9]:
def extract_augmented_train_features(df_train, cfg, augmenter, desc="Augment train"):
    """
    Augmentasi hanya untuk split train.
    Return X_aug dan meta_aug.
    """
    if augmenter is None or cfg["aug_n_copies"] <= 0:
        return (
            None,
            pd.DataFrame(),
            pd.DataFrame()
        )

    X_aug = []
    rows_aug = []
    errors = []

    for idx, row in tqdm(df_train.iterrows(), total=len(df_train), desc=desc):
        for aug_idx in range(cfg["aug_n_copies"]):
            try:
                audio = load_audio(
                    row["filepath"],
                    sr=cfg["sample_rate"],
                    duration=cfg["duration"]
                )

                audio_aug = augmenter(
                    samples=audio,
                    sample_rate=cfg["sample_rate"]
                ).astype(np.float32)

                feat_aug = extract_handcrafted(audio_aug, cfg)

                new_row = row.to_dict()
                new_row["uid"] = f"{row['uid']}_aug{aug_idx}"
                new_row["is_augmented"] = True
                new_row["augmentation_index"] = aug_idx
                new_row["original_uid"] = row["uid"]

                X_aug.append(feat_aug)
                rows_aug.append(new_row)

            except Exception as e:
                errors.append({
                    "idx": idx,
                    "filepath": row["filepath"],
                    "aug_idx": aug_idx,
                    "error": str(e)
                })

    if len(X_aug) > 0:
        X_aug = np.asarray(X_aug, dtype=np.float32)
    else:
        X_aug = None

    meta_aug = pd.DataFrame(rows_aug)
    errors_aug = pd.DataFrame(errors)

    return X_aug, meta_aug, errors_aug

In [10]:
def process_one_dataset(dataset_name, csv_path, cfg, out_root, augmenter=None):
    print("=" * 90)
    print(f"Processing dataset: {dataset_name.upper()}")
    print("=" * 90)

    dataset_out = out_root / dataset_name
    dataset_out.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(csv_path)

    # Safety checks
    required_cols = ["uid", "filepath", "dataset", "emotion", "label", "speaker", "split"]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing columns in {csv_path}: {missing_cols}")

    df = df[df["emotion"].isin(cfg["unified_emotions"])].copy()
    df["label"] = df["emotion"].map(LABEL_TO_ID).astype(int)
    df["is_augmented"] = False
    df["augmentation_index"] = -1
    df["original_uid"] = df["uid"]

    print("Rows:", len(df))
    print("Speakers:", df["speaker"].nunique())
    print("\nDistribution:")
    display(df.groupby(["split", "emotion"]).size().unstack(fill_value=0))

    df_train = df[df["split"] == "train"].reset_index(drop=True)
    df_val = df[df["split"] == "val"].reset_index(drop=True)
    df_test = df[df["split"] == "test"].reset_index(drop=True)

    # 1. Original features
    X_train_orig, meta_train_orig, err_train = extract_original_features(
        df_train, cfg, desc=f"{dataset_name} train original"
    )
    X_val, meta_val, err_val = extract_original_features(
        df_val, cfg, desc=f"{dataset_name} val original"
    )
    X_test, meta_test, err_test = extract_original_features(
        df_test, cfg, desc=f"{dataset_name} test original"
    )

    print("Original shapes:")
    print("X_train_orig:", X_train_orig.shape)
    print("X_val       :", X_val.shape)
    print("X_test      :", X_test.shape)

    # 2. Train augmentation
    X_train_aug, meta_train_aug, err_aug = extract_augmented_train_features(
        meta_train_orig,
        cfg,
        augmenter,
        desc=f"{dataset_name} train augmentation"
    )

    if X_train_aug is None:
        print("Augmented shape: None, augmentation skipped")
    else:
        print("Augmented shape:", X_train_aug.shape)

    # 3. Combine train original + augmented
    if X_train_aug is not None and len(X_train_aug) > 0:
        X_train_raw = np.concatenate([X_train_orig, X_train_aug], axis=0)
        meta_train = pd.concat([meta_train_orig, meta_train_aug], ignore_index=True)
    else:
        X_train_raw = X_train_orig
        meta_train = meta_train_orig.copy()

    y_train = meta_train["label"].values.astype(np.int64)
    y_val = meta_val["label"].values.astype(np.int64)
    y_test = meta_test["label"].values.astype(np.int64)

    # 4. Fit scaler only on train
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    # 5. Save
    np.save(dataset_out / "X_hc_train.npy", X_train)
    np.save(dataset_out / "y_train.npy", y_train)

    np.save(dataset_out / "X_hc_val.npy", X_val)
    np.save(dataset_out / "y_val.npy", y_val)

    np.save(dataset_out / "X_hc_test.npy", X_test)
    np.save(dataset_out / "y_test.npy", y_test)

    meta_train.to_csv(dataset_out / "meta_train.csv", index=False)
    meta_val.to_csv(dataset_out / "meta_val.csv", index=False)
    meta_test.to_csv(dataset_out / "meta_test.csv", index=False)

    error_df = pd.concat(
        [
            err_train.assign(stage="train_original"),
            err_val.assign(stage="val_original"),
            err_test.assign(stage="test_original"),
            err_aug.assign(stage="train_augmentation"),
        ],
        ignore_index=True
    )
    error_df.to_csv(dataset_out / "errors.csv", index=False)

    with open(dataset_out / "scaler_hc.pkl", "wb") as f:
        pickle.dump(scaler, f)

    feature_config = {
        "dataset": dataset_name,
        "sample_rate": cfg["sample_rate"],
        "duration": cfg["duration"],
        "n_mfcc": cfg["n_mfcc"],
        "n_mels": cfg["n_mels"],
        "n_fft": cfg["n_fft"],
        "hop_length": cfg["hop_length"],
        "aug_n_copies": cfg["aug_n_copies"],
        "feature_dim": int(X_train.shape[1]),
        "labels": cfg["unified_emotions"],
        "label_to_id": LABEL_TO_ID,
        "scaler_fit_on": "train_original_plus_train_augmented_only",
        "val_test_augmented": False,
    }

    with open(dataset_out / "feature_config.json", "w") as f:
        json.dump(feature_config, f, indent=2)

    print("\nSaved to:", dataset_out)
    print("Final shapes:")
    print("X_train:", X_train.shape, "y_train:", y_train.shape)
    print("X_val  :", X_val.shape, "y_val  :", y_val.shape)
    print("X_test :", X_test.shape, "y_test :", y_test.shape)

    print("\nTrain distribution after augmentation:")
    display(meta_train.groupby(["is_augmented", "emotion"]).size().unstack(fill_value=0))

    print("\nVal distribution:")
    display(meta_val["emotion"].value_counts().reindex(cfg["unified_emotions"], fill_value=0))

    print("\nTest distribution:")
    display(meta_test["emotion"].value_counts().reindex(cfg["unified_emotions"], fill_value=0))

    if len(error_df) > 0:
        print("\n⚠️ Errors:", len(error_df))
        display(error_df.head())
    else:
        print("\n✅ No extraction errors")

    return {
        "dataset": dataset_name,
        "out_dir": str(dataset_out),
        "X_train_shape": X_train.shape,
        "X_val_shape": X_val.shape,
        "X_test_shape": X_test.shape,
        "n_errors": len(error_df),
    }

In [11]:
results = []

for dataset_name, csv_path in CONFIG["csv_files"].items():
    result = process_one_dataset(
        dataset_name=dataset_name,
        csv_path=csv_path,
        cfg=CONFIG,
        out_root=OUT_DIR,
        augmenter=augmenter
    )
    results.append(result)

results_df = pd.DataFrame(results)
display(results_df)

results_df.to_csv(OUT_DIR / "feature_extraction_summary.csv", index=False)
print("Saved summary:", OUT_DIR / "feature_extraction_summary.csv")

Processing dataset: EMODB
Rows: 718
Speakers: 10

Distribution:


emotion,angry,disgust,fear,happy,neutral,sad
split,,,,,,
test,29,22,25,24,22,27
train,98,73,85,82,74,86
val,13,11,13,12,10,12


emodb test original: 100%|██████████| 149/149 [03:30<00:00,  1.41s/it]


Original shapes:
X_train_orig: (498, 548)
X_val       : (71, 548)
X_test      : (149, 548)
Augmented shape: None, augmentation skipped

Saved to: /content/drive/MyDrive/New Jurnal Cross/processed_intra_features_hc_noaug/emodb
Final shapes:
X_train: (498, 548) y_train: (498,)
X_val  : (71, 548) y_val  : (71,)
X_test : (149, 548) y_test : (149,)

Train distribution after augmentation:


emotion,angry,disgust,fear,happy,neutral,sad
is_augmented,,,,,,
False,98,73,85,82,74,86


...


Test distribution:


,count
emotion,
angry,34
disgust,15
fear,29
happy,23
neutral,20
sad,18



✅ No extraction errors


,dataset,out_dir,X_train_shape,X_val_shape,X_test_shape,n_errors
0,emodb,/content/drive/MyDrive/New Jurnal Cross/proces...,"(498, 548)","(71, 548)","(149, 548)",0
1,ravdess,/content/drive/MyDrive/New Jurnal Cross/proces...,"(704, 548)","(176, 548)","(176, 548)",0
2,resd,/content/drive/MyDrive/New Jurnal Cross/proces...,"(873, 548)","(186, 548)","(139, 548)",0


Saved summary: /content/drive/MyDrive/New Jurnal Cross/processed_intra_features_hc_noaug/feature_extraction_summary.csv


In [12]:
for dataset_name in CONFIG["csv_files"].keys():
    ds_dir = OUT_DIR / dataset_name

    print("=" * 80)
    print(dataset_name.upper())

    X_train = np.load(ds_dir / "X_hc_train.npy")
    y_train = np.load(ds_dir / "y_train.npy")

    X_val = np.load(ds_dir / "X_hc_val.npy")
    y_val = np.load(ds_dir / "y_val.npy")

    X_test = np.load(ds_dir / "X_hc_test.npy")
    y_test = np.load(ds_dir / "y_test.npy")

    meta_train = pd.read_csv(ds_dir / "meta_train.csv")
    meta_val = pd.read_csv(ds_dir / "meta_val.csv")
    meta_test = pd.read_csv(ds_dir / "meta_test.csv")

    print("X_train:", X_train.shape, "y_train:", y_train.shape, "meta_train:", meta_train.shape)
    print("X_val  :", X_val.shape, "y_val  :", y_val.shape, "meta_val  :", meta_val.shape)
    print("X_test :", X_test.shape, "y_test :", y_test.shape, "meta_test :", meta_test.shape)

    print("Any NaN train:", np.isnan(X_train).any())
    print("Any NaN val  :", np.isnan(X_val).any())
    print("Any NaN test :", np.isnan(X_test).any())

    print("Feature dim:", X_train.shape[1])

    print("\nTrain labels:")
    print(pd.Series(y_train).value_counts().sort_index().rename(index=ID_TO_LABEL))

    print("\nVal labels:")
    print(pd.Series(y_val).value_counts().sort_index().rename(index=ID_TO_LABEL))

    print("\nTest labels:")
    print(pd.Series(y_test).value_counts().sort_index().rename(index=ID_TO_LABEL))

EMODB
X_train: (498, 548) y_train: (498,) meta_train: (498, 31)
X_val  : (71, 548) y_val  : (71,) meta_val  : (71, 31)
X_test : (149, 548) y_test : (149,) meta_test : (149, 31)
Any NaN train: False
Any NaN val  : False
Any NaN test : False
Feature dim: 548

Train labels:
angry      98
disgust    73
fear       85
happy      82
neutral    74
sad        86
Name: count, dtype: int64

Val labels:
angry      13
disgust    11
fear       13
happy      12
neutral    10
sad        12
Name: count, dtype: int64

Test labels:
angry      29
disgust    22
fear       25
happy      24
neutral    22
sad        27
Name: count, dtype: int64
RAVDESS
X_train: (704, 548) y_train: (704,) meta_train: (704, 31)
X_val  : (176, 548) y_val  : (176,) meta_val  : (176, 31)
X_test : (176, 548) y_test : (176,) meta_test : (176, 31)
Any NaN train: False
Any NaN val  : False
Any NaN test : False
Feature dim: 548

Train labels:
angry      128
disgust    128
fear       128
happy      128
neutral     64
sad        128
Name